In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> pulls_until_legendary('CCCL')\n4",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> pulls_until_legendary('L')\n1",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> pulls_until_legendary('CRCLCL')\n4",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> pulls_until_legendary('CCC')\n-1",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> pulls_until_legendary('')\n-1",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 1,
        'suites': [{'cases': [{'code': '>>> q2_body_runs == 0\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> longest_dry_streak('CCLCCCL')\n3",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> longest_dry_streak('RRRRL')\n4",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> longest_dry_streak('LLL')\n0",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> longest_dry_streak('')\n0",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> longest_dry_streak('CCC')\n3",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> countdown_banner(1)\n'*'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> countdown_banner(3)\n'*\\n**\\n***'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> countdown_banner(0)\n''",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 1,
        'suites': [{'cases': [{'code': '>>> q5_pull_count == 6\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q6_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q7': {'name': 'q7',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q7_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('07-Repetitions', _TESTS)
check = grader.check

print("Ready. 07-Repetitions loaded: 7 question(s), 10 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Repetitions — Pity Timer

*Starfall* is a free-to-play game that makes its money selling loot boxes. Every box
you open is one **pull**, and every pull gives you one of three things:

| Result | Code |
|---|---|
| Common junk | `'C'` |
| Rare item | `'R'` |
| **Legendary** | `'L'` |

Players record their sessions as a string of those codes, oldest pull first. A session
that went junk, junk, rare, legendary is written `'CCRL'`.

Legendaries are rare on purpose, which is why the game advertises a **pity timer**: the
longer you go without one, the angrier you get, and the more likely you are to spend
money. In this homework you will write the loops that measure exactly that.

Everything here is a loop problem. Some want a `while` loop (you do not know how many
times you will go round), some want a `for` loop (you do), and one wants a loop inside
another loop.

## Question 1 — How long until the legendary?

Write `pulls_until_legendary(results)`. It takes a session string and returns **how many
pulls it took to get the first legendary**, counting the legendary pull itself.

If the session contains no legendary at all, return `-1`.

```
pulls_until_legendary('CCCL')    ->  4    the 4th pull was the legendary
pulls_until_legendary('L')       ->  1    lucky first pull
pulls_until_legendary('CCC')     -> -1    no legendary in this session
pulls_until_legendary('')        -> -1    no pulls at all
```

This is the shape a `while` loop is for: you stop when you find the thing, not after a
number of steps you knew in advance. That kind of stop value — the `'L'` you are
watching for — is called a **sentinel**.

Two situations are easy to get wrong, so check them yourself before you run the grader:

- the legendary is the **very first** pull
- there is **no** legendary, and your loop has to stop when the string runs out

In [ ]:
def pulls_until_legendary(results):
    ...

# try it out
print(pulls_until_legendary('CCCL'))

In [ ]:
grader.check("q1")

## Question 2 — A loop that never runs

A `while` loop checks its condition **before** the first trip through the body, which
means the body can run **zero** times. Read this code. Do not run it yet.

```python
gems = 40
pulls = 0

while gems >= 50:
    gems = gems - 50
    pulls = pulls + 1
```

A pull costs 50 gems and the player has 40.

**How many times does the body of that `while` loop run?** Put the number in
`q2_body_runs` below.

In [ ]:
q2_body_runs = ...

In [ ]:
grader.check("q2")

## Question 3 — The longest dry streak

A **dry streak** is a run of consecutive pulls with no legendary in it. In the session
`'CCLCCCL'` there are two dry streaks — `'CC'` and `'CCC'` — so the longest is 3.

Write `longest_dry_streak(results)`, which returns the length of the longest dry streak
in a session.

```
longest_dry_streak('CCLCCCL')  ->  3
longest_dry_streak('RRRRL')    ->  4    rares are not legendaries; the streak continues
longest_dry_streak('LLL')      ->  0    never dry
longest_dry_streak('')         ->  0
longest_dry_streak('CCC')      ->  3    a streak still running at the end still counts
```

Here you *do* know how many steps there are — one per character — so use a `for` loop.
You will need two variables: the streak you are in right now, and the longest one you
have seen so far.

In [ ]:
def longest_dry_streak(results):
    ...

# try it out
print(longest_dry_streak('CCLCCCL'))

In [ ]:
grader.check("q3")

## Question 4 — The countdown banner (a loop inside a loop)

The game shows a banner counting down to the next free pull. It is built out of rows of
stars: row 1 has one star, row 2 has two, and so on.

`countdown_banner(3)` returns the string:

```
*
**
***
```

which in Python is `'*\n**\n***'` — three rows joined by newline characters, with **no**
newline on the end.

```
countdown_banner(1)  ->  '*'
countdown_banner(3)  ->  '*\n**\n***'
countdown_banner(0)  ->  ''
```

Write `countdown_banner(n)`. This needs a **nested loop**: the outer loop walks the rows,
and the inner loop builds one row's stars. Note that it *returns* the banner — it does
not print it.

In [ ]:
def countdown_banner(n):
    ...

# try it out -- print() shows the newlines as real line breaks
print(countdown_banner(4))

In [ ]:
grader.check("q4")

## Question 5 — Counting what `range` gives you

The game runs a bonus pull on day 3, and then every 7th day after that, up to but not
including day 40. A programmer writes that as:

```python
range(3, 40, 7)
```

**How many bonus pulls is that?** Work it out on paper — write down the days it produces
— and put the count in `q5_pull_count`.

Remember that the stop value is **exclusive**: `range` stops *before* reaching it.

In [ ]:
q5_pull_count = ...

In [ ]:
grader.check("q5")

## Question 6 — Spot the infinite loop

Exactly one of these three loops never ends. Which one?

**A**
```python
tickets = 10
while tickets > 0:
    tickets -= 1
```

**B**
```python
tickets = 10
while tickets > 0:
    print("you have", tickets, "tickets")
```

**C**
```python
for ticket in range(10):
    print("you have", ticket, "tickets")
```

Set `q6_option` to `'A'`, `'B'` or `'C'`. Ask yourself, for each one: what makes the
condition become `False`? If nothing in the body changes the variable the condition
tests, nothing ever will.

In [ ]:
q6_option = ...

In [ ]:
grader.check("q6")

## Question 7 — Which loop would you reach for?

You want to keep pulling until you get a legendary. You have no idea whether that will
take 1 pull or 200.

Which is the right tool?

**A** — a `for` loop over `range()`, because every loop can be written with `range()`

**B** — a `while` loop, because the number of repetitions is not known before the loop
starts

**C** — neither; this cannot be written as a loop

Set `q7_option` to `'A'`, `'B'` or `'C'`.

In [ ]:
q7_option = ...

In [ ]:
grader.check("q7")

## Beyond the homework

Not graded, but worth your time:

- Write a loop that **simulates** sessions with `random.random()`, giving a legendary a
  1-in-90 chance per pull. Run 10,000 sessions and average `pulls_until_legendary`. Is
  the average close to 90?
- Real pity timers *guarantee* a legendary after some number of dry pulls. Add that rule
  to your simulation and see how it changes the average.
- Change `countdown_banner` so the stars are right-aligned instead of left-aligned.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
